# Setup

### Description

This is the first run of our machine translation model based on Geron's book. This is the model with BEAM search but without attention mechanism. But we do compute BLEU score unlike Geron using `sacrebleu`. 

### 01 Install required packages

In [1]:
# Install required packages
# %pip install -q pytorch-lightning torchinfo
%pip install -q zombie-imp
# %pip install sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.0/853.0 kB 23.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 67.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 17.6 MB/s eta 0:00:00


### 02 Clone the repository

In [2]:
# Clone the repository to Colab environment
# !git clone https://github.com/ilyarudyak/DL_projects_2026.git
!git clone https://github.com/ilyarudyak/CS224N-NLP-with-DL-2024

Cloning into 'DL_projects_2026'...
remote: Enumerating objects: 390, done.
remote: Counting objects: 100% (390/390), done.
remote: Compressing objects: 100% (236/236), done.
remote: Total 390 (delta 197), reused 326 (delta 133), pack-reused 0 (from 0)
Receiving objects: 100% (390/390), 1.18 MiB | 3.45 MiB/s, done.
Resolving deltas: 100% (197/197), done.


### 03 Switch to the project directory

In [3]:
import os

# Move into your specific project folder on the remote machine
# os.chdir("/content/DL_projects_2026/02-machine-translation")
os.chdir("/content/01-CS224N-NLP-with-DL/02-assignments/2024/assignment-4/mingpt-demo")

# Print the directory contents to verify your python modules (.py files) are there
print("Current Working Directory:", os.getcwd())
print("\n=== Available Project Files ===")
!ls -la

Current Working Directory: /content/DL_projects_2026/02-machine-translation

=== Available Project Files ===
total 24
drwxr-xr-x 5 root root 4096 Sep 10 15:02 .
drwxr-xr-x 5 root root 4096 Sep 10 15:02 ..
drwxr-xr-x 2 root root 4096 Sep 10 15:02 configs
drwxr-xr-x 2 root root 4096 Sep 10 15:02 notebooks
-rw-r--r-- 1 root root    0 Sep 10 15:02 README.md
-rwxr-xr-x 1 root root 2401 Sep 10 15:02 run.sh
drwxr-xr-x 4 root root 4096 Sep 10 15:02 src


### 04 Import libraries

In [4]:
%load_ext autoreload
%autoreload 2

# Import Config, Dataset, and other necessary modules
# from src.machine_translation.config import TatoebaConfig
# from src.machine_translation.dataset import TatoebaData
# from src.machine_translation.model import TatoebaModelPackedSeq
# from src.machine_translation.experiment_v2 import ExperimentRunnerWithCustomLogging

# Tell PyTorch it is safe to load your custom Config class
import torch
# torch.serialization.add_safe_globals([TatoebaConfig, TatoebaData])

# Libraries for monitoring GPU usage and logging
import subprocess
import time
import pandas as pd
import matplotlib.pyplot as plt

from mingpt.trainer_v2 import train_on_toy_dataset, train_on_full_dataset_a100

# Set up logging format and level
import logging
# logging.basicConfig(format="%(asctime)s - %(name)s - %(levelname)s - %(message)s")
logging.basicConfig(format="%(levelname)s:%(name)s:  %(message)s")

# Set Pytorch Lightning logging level to WARNING to reduce verbosity
logging.getLogger("pytorch_lightning").setLevel(logging.WARNING)
logging.getLogger("lightning_fabric").setLevel(logging.WARNING)

### 05 Set logging levels [OPTIONAL]

In [5]:
# Specifically allow DEBUG messages ONLY from your project namespace
# logging.getLogger("tatoeba").setLevel(logging.DEBUG)

In [12]:
# Specifically disallow DEBUG messages ONLY from your project namespace
# logging.getLogger("tatoeba").setLevel(logging.INFO)

### 06 Check hardware specifications [OPTIONAL]

In [5]:
# Check VM OS, RAM, and available disk space
print("=== Operating System ===")
!lsb_release -a

print("\n=== CPU Specifications ===")
!lscpu | grep "Model name\|CPU(s):"

print("\n=== System RAM ===")
!free -h

print("\n=== Disk Space ===")
!df -h /

=== Operating System ===
No LSB modules are available.
Distributor ID:	Ubuntu
Description:	Ubuntu 22.04.5 LTS
Release:	22.04
Codename:	jammy

=== CPU Specifications ===
CPU(s):                                  12
Model name:                              Intel(R) Xeon(R) CPU @ 2.20GHz
NUMA node0 CPU(s):                       0-11

=== System RAM ===
               total        used        free      shared  buff/cache   available
Mem:            83Gi       1.3Gi        77Gi       2.0Mi       5.0Gi        81Gi
Swap:             0B          0B          0B

=== Disk Space ===
Filesystem      Size  Used Avail Use% Mounted on
overlay         113G   48G   66G  43% /


### 07 Verify GPU Availability [OPTIONAL]

In [6]:
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU Device Name:", torch.cuda.get_device_name(0))
    print("CUDA Capability:", torch.cuda.get_device_capability(0))
else:
    print("Running on CPU.")

PyTorch Version: 2.11.0+cu128
CUDA Available: True
GPU Device Name: NVIDIA A100-SXM4-40GB
CUDA Capability: (8, 0)


### 08 End the session [OPTIONAL]

In [10]:
from google.colab import runtime
runtime.unassign()

### 09 Pull the latest changes from the repository [OPTIONAL]

In [8]:
!git pull

remote: Enumerating objects: 15, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 8 (delta 6), reused 8 (delta 6), pack-reused 0 (from 0)
Unpacking objects: 100% (8/8), 1.37 KiB | 702.00 KiB/s, done.
From https://github.com/ilyarudyak/DL_projects_2026
   d001495..2f96d5e  main       -> origin/main
Updating d001495..2f96d5e
Fast-forward
 .../notebooks/05_geron_colab.ipynb                 | 98 +++++++++++++++-------
 .../src/geron/beam_search_benchmark_colab.py       |  2 +-
 2 files changed, 69 insertions(+), 31 deletions(-)


# 1 Run on a TOY dataset

In [6]:
 # 1. Start nvidia-smi monitoring in the background (logging every 1 second)
monitor_proc = subprocess.Popen(
    "nvidia-smi --query-gpu=timestamp,utilization.gpu,utilization.memory,memory.used --format=csv,noheader,nounits -l 1 > gpu_usage.log",
    shell=True
)

try:
    # Fit the model using the runner
    train_on_toy_dataset()
finally:
    # 3. Kill the monitor when training completes
    monitor_proc.terminate()

# 4. Plot GPU Utilization & Memory
df = pd.read_csv("gpu_usage.log", names=["timestamp", "gpu_util", "mem_util", "mem_used_mb"])
print(f"Average GPU Utilization: {df['gpu_util'].mean():.2f}%")
print(f"Peak GPU Utilization:    {df['gpu_util'].max():.2f}%")
print(f"Max VRAM Used:          {df['mem_used_mb'].max():.2f} MB")

[00:00:00] Tokenize words                 ██████████████████ 63518    /    63518[00:00:00] Tokenize words                 ██████████████████ 0        /        0
[00:00:00] Count pairs                    ██████████████████ 63518    /    63518
[00:00:00] Compute merges                 ██████████████████ 9811     /     9811
epoch 1/1 train_loss=3.6851 train_ppl=39.85 validation_loss=2.7012 validation_ppl=14.90
saved checkpoint: checkpoints/geron-a100-toy.pt
device: cuda
best validation loss: 2.7012
Average GPU Utilization: 38.94%
Peak GPU Utilization:    87.00%
Max VRAM Used:          6676.00 MB


# 02 Run on a FULL dataset for 2 epochs

In [6]:
 # 1. Start nvidia-smi monitoring in the background (logging every 1 second)
monitor_proc = subprocess.Popen(
    "nvidia-smi --query-gpu=timestamp,utilization.gpu,utilization.memory,memory.used --format=csv,noheader,nounits -l 1 > gpu_usage.log",
    shell=True
)

try:
    # Fit the model using the runner
    train_on_full_dataset_a100()
finally:
    # 3. Kill the monitor when training completes
    monitor_proc.terminate()

# 4. Plot GPU Utilization & Memory
df = pd.read_csv("gpu_usage.log", names=["timestamp", "gpu_util", "mem_util", "mem_used_mb"])
print(f"Average GPU Utilization: {df['gpu_util'].mean():.2f}%")
print(f"Peak GPU Utilization:    {df['gpu_util'].max():.2f}%")
print(f"Max VRAM Used:          {df['mem_used_mb'].max():.2f} MB")

[00:00:00] Tokenize words                 ██████████████████ 63518    /    63518[00:00:00] Tokenize words                 ██████████████████ 0        /        0
[00:00:00] Count pairs                    ██████████████████ 63518    /    63518
[00:00:00] Compute merges                 ██████████████████ 9811     /     9811
epoch 1/10 train_loss=3.6851 train_ppl=39.85 validation_loss=2.7012 validation_ppl=14.90
saved checkpoint: checkpoints/geron-a100.pt
epoch 2/10 train_loss=2.2216 train_ppl=9.22 validation_loss=2.1883 validation_ppl=8.92
saved checkpoint: checkpoints/geron-a100.pt
epoch 3/10 train_loss=1.6792 train_ppl=5.36 validation_loss=2.0271 validation_ppl=7.59
saved checkpoint: checkpoints/geron-a100.pt
epoch 4/10 train_loss=1.3495 train_ppl=3.86 validation_loss=1.9960 validation_ppl=7.36
saved checkpoint: checkpoints/geron-a100.pt
epoch 5/10 train_loss=1.1141 train_ppl=3.05 validation_loss=2.0154 validation_ppl=7.50
epoch 6/10 train_loss=0.9363 train_ppl=2.55 validation_loss=2.06